# Dose–Response Logistic Regression: Budworm Mortality

**Academic binary-outcome optimization case study — self-contained notebook**

Twenty male and twenty female tobacco budworms were exposed at each of six doses of the pyrethroid insecticide *trans*-cypermethrin. The binary event is **killed (1) / alive (0)**. The 12 grouped rows represent 240 Bernoulli trials.

This classic dataset appears in Venables & Ripley's *Modern Applied Statistics* and Collett's *Modelling Binary Data*. The values are reproduced in the official [R/MASS `dose.p` documentation](https://stat.ethz.ch/R-manual/R-devel/RHOME/library/MASS/html/dose.p.html) and the [CRAN GLMsData documentation](https://stat.ethz.ch/CRAN/web/packages/GLMsData/GLMsData.pdf).

**Goals:** derive the grouped-binomial likelihood, gradient, and Hessian; study convexity and separation; implement gradient descent and damped Newton with Armijo backtracking; compare with SciPy BFGS and trust-exact; estimate dose–response curves and LD50 with uncertainty; and distinguish numerical convergence from statistical adequacy.

## 1. Statistical experiment and model

For experimental group $i$, let $K_i$ be the number killed among $n_i=20$ exposed insects:

$$K_i\mid x_i,s_i\sim\operatorname{Binomial}(n_i,p_i).$$

Define standardized log-dose

$$x_i=\frac{\log_2(d_i)-\bar x}{s_x},$$

and $s_i=1$ for female, $0$ for male. We fit the interaction model

$$
\operatorname{logit}(p_i)=\eta_i
=\beta_0+\beta_1x_i+\beta_2s_i+\beta_3x_is_i.
$$

Thus males and females may have different intercepts and dose slopes. Standardizing log-dose leaves fitted probabilities unchanged but improves the conditioning of gradient-based optimization. Conditional independence, binomial sampling within groups, correct logit specification, and absence of unmodelled overdispersion are the maintained assumptions.

## 2. Likelihood, derivatives, and convexity

Ignoring the binomial coefficients, the negative log-likelihood is

$$
J(\beta)=\sum_{i=1}^{m}\left[n_i\log(1+e^{\eta_i})-K_i\eta_i\right],
\qquad \eta=X\beta.
$$

With $p_i=\sigma(\eta_i)$, $W=\operatorname{diag}\{n_ip_i(1-p_i)\}$, and $K=(K_1,\ldots,K_m)^\top$,

$$
\nabla J(\beta)=X^\top(np-K),\qquad
\nabla^2J(\beta)=X^\top W X.
$$

For every vector $v$,

$$v^\top\nabla^2Jv=(Xv)^\top W(Xv)\ge0,$$

so the objective is convex. It is strictly convex when $X$ has full column rank and the fitted probabilities remain interior. Under complete or quasi-complete separation, the unpenalized finite MLE may fail to exist and Hessian conditioning deteriorates. A ridge term $\lambda\|\beta_{1:}\|_2^2/2$ would restore coercivity; the primary fit here uses $\lambda=0$ because this experiment has a finite, well-conditioned MLE.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit, gammaln
from scipy.stats import chi2, norm

np.set_printoptions(precision=6, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

data = pd.read_csv(Path("data/budworm_dose_mortality.csv"))
assert (data["killed"] + data["alive"] == data["exposed"]).all()
assert data["exposed"].sum() == 240

log_dose_mean = data["log2_dose"].mean()
log_dose_scale = data["log2_dose"].std(ddof=0)
x_standardized = (data["log2_dose"] - log_dose_mean) / log_dose_scale
female = (data["gender"] == "Female").astype(float)
X = np.column_stack([
    np.ones(len(data)), x_standardized, female, x_standardized * female
])
killed = data["killed"].to_numpy(float)
exposed = data["exposed"].to_numpy(float)
parameter_names = ["intercept", "standardized_log2_dose", "female", "dose_x_female"]

display(data)
fig, ax = plt.subplots(figsize=(8, 4.5))
for gender, group in data.groupby("gender"):
    ax.plot(group["dose_micrograms"], group["killed"]/group["exposed"],
            "o-", label=gender)
ax.set(xscale="log", xlabel="dose (micrograms; log scale)",
       ylabel="observed mortality proportion", title="Budworm dose–mortality experiment")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
RIDGE = 0.0
R = np.diag([0.0, 1.0, 1.0, 1.0])

def objective(beta, design=X, events=killed, totals=exposed, ridge=RIDGE):
    eta = design @ beta
    return float(np.sum(totals*np.logaddexp(0.0, eta) - events*eta)
                 + 0.5*ridge*beta@R@beta)


def derivatives(beta, design=X, events=killed, totals=exposed, ridge=RIDGE):
    eta = design @ beta
    p = expit(eta)
    gradient = design.T @ (totals*p-events) + ridge*R@beta
    weights = totals*p*(1-p)
    hessian = design.T @ (design*weights[:, None]) + ridge*R
    return gradient, hessian


def finite_difference_gradient(fun, beta, relative_step=1e-6):
    result = np.empty_like(beta, dtype=float)
    for j in range(len(beta)):
        h = relative_step*max(1.0, abs(beta[j]))
        forward, backward = beta.copy(), beta.copy()
        forward[j] += h; backward[j] -= h
        result[j] = (fun(forward)-fun(backward))/(2*h)
    return result


audit_point = np.array([-0.4, 1.2, -0.3, 0.2])
analytic_gradient, audit_hessian = derivatives(audit_point)
numeric_gradient = finite_difference_gradient(objective, audit_point)
relative_error = np.linalg.norm(analytic_gradient-numeric_gradient) / max(
    1.0, np.linalg.norm(numeric_gradient)
)
print("Design rank:", np.linalg.matrix_rank(X), "/", X.shape[1])
print("Hessian eigenvalues at audit point:", np.linalg.eigvalsh(audit_hessian))
print(f"Relative gradient error: {relative_error:.3e}")
assert np.linalg.matrix_rank(X) == X.shape[1]
assert relative_error < 1e-7

## 3. Optimization algorithms and stopping conditions

**Gradient descent** uses $d_k=-\nabla J(\beta_k)$. **Newton's method** solves

$$\nabla^2J(\beta_k)d_k=-\nabla J(\beta_k).$$

Both use Armijo backtracking, accepting the first $a_k\in\{1,1/2,1/4,\ldots\}$ satisfying

$$J(\beta_k+a_kd_k)\le J(\beta_k)+c_1a_k\nabla J(\beta_k)^\top d_k.$$

Stopping requires $\|\nabla J\|_\infty\le10^{-8}$, a negligible step, line-search failure, or the iteration limit. Newton has local quadratic convergence when the Hessian is nonsingular and the solution lies in a regular region. Backtracking globalizes the iterations for this convex problem but cannot rescue an unpenalized model whose MLE is at infinity because of separation.

In [ ]:
def armijo_search(beta, direction, value, gradient, c1=1e-4, contraction=.5):
    directional = float(gradient @ direction)
    step = 1.0
    while step >= 1e-14:
        candidate = beta + step*direction
        if objective(candidate) <= value + c1*step*directional:
            return step, candidate
        step *= contraction
    raise RuntimeError("Armijo line search failed")


def gradient_descent(beta0, gtol=1e-8, max_iter=20000):
    beta = np.asarray(beta0, dtype=float).copy()
    history = []
    for iteration in range(max_iter+1):
        value = objective(beta)
        gradient, _ = derivatives(beta)
        gradient_norm = np.linalg.norm(gradient, np.inf)
        history.append((iteration, value, gradient_norm))
        if gradient_norm <= gtol:
            return beta, pd.DataFrame(history, columns=["iteration","objective","gradient_inf"]), True
        step, candidate = armijo_search(beta, -gradient, value, gradient)
        if np.linalg.norm(candidate-beta) <= 1e-12*(1+np.linalg.norm(beta)):
            return candidate, pd.DataFrame(history, columns=["iteration","objective","gradient_inf"]), True
        beta = candidate
    return beta, pd.DataFrame(history, columns=["iteration","objective","gradient_inf"]), False


def damped_newton(beta0, gtol=1e-8, max_iter=100):
    beta = np.asarray(beta0, dtype=float).copy()
    history = []
    for iteration in range(max_iter+1):
        value = objective(beta)
        gradient, hessian = derivatives(beta)
        gradient_norm = np.linalg.norm(gradient, np.inf)
        history.append((iteration, value, gradient_norm))
        if gradient_norm <= gtol:
            return beta, pd.DataFrame(history, columns=["iteration","objective","gradient_inf"]), True
        direction = -np.linalg.solve(hessian, gradient)
        _, beta = armijo_search(beta, direction, value, gradient)
    return beta, pd.DataFrame(history, columns=["iteration","objective","gradient_inf"]), False


beta0 = np.zeros(X.shape[1])
beta_gd, history_gd, success_gd = gradient_descent(beta0)
beta_newton, history_newton, success_newton = damped_newton(beta0)
reference_bfgs = minimize(
    objective, beta0, jac=lambda b: derivatives(b)[0], method="BFGS",
    options={"gtol": 1e-7, "maxiter": 1000}
)
reference_trust = minimize(
    objective, beta0, jac=lambda b: derivatives(b)[0],
    hess=lambda b: derivatives(b)[1], method="trust-exact",
    options={"gtol": 1e-10, "maxiter": 1000}
)

solutions = [
    ("Gradient descent + Armijo", beta_gd, success_gd, len(history_gd)-1),
    ("Damped Newton + Armijo", beta_newton, success_newton, len(history_newton)-1),
    ("SciPy BFGS reference", reference_bfgs.x, reference_bfgs.success, reference_bfgs.nit),
    ("SciPy trust-exact reference", reference_trust.x, reference_trust.success, reference_trust.nit),
]
solver_table = pd.DataFrame([{
    "method": name, "objective": objective(beta),
    "gradient_inf_norm": np.linalg.norm(derivatives(beta)[0], np.inf),
    "iterations": iterations, "success": success,
    "distance_to_BFGS": np.linalg.norm(beta-reference_bfgs.x)
} for name, beta, success, iterations in solutions])
display(solver_table)

coefficient_table = pd.DataFrame({
    "parameter": parameter_names, "estimate": beta_newton
})
display(coefficient_table)
assert success_gd and success_newton and reference_trust.success
assert np.linalg.norm(beta_newton-reference_bfgs.x) < 1e-6

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for history, name in [(history_gd, "Gradient descent"), (history_newton, "Newton")]:
    axes[0].semilogy(history["iteration"], history["gradient_inf"], label=name)
    axes[1].plot(history["iteration"], history["objective"]-objective(beta_newton)+1e-14,
                 label=name)
axes[0].axhline(1e-8, color="black", ls="--", lw=1)
axes[0].set(xlabel="iteration", ylabel=r"$\|\nabla J\|_\infty$", title="First-order convergence")
axes[1].set(yscale="log", xlabel="iteration", ylabel="objective gap", title="Objective convergence")
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()

## 4. Inference, LD50, and scientific interpretation

At the MLE, model-based covariance is

$$\widehat{\operatorname{Cov}}(\widehat\beta)=
[X^\top\widehat W X]^{-1}.$$

For sex indicator $s\in\{0,1\}$, the standardized log-dose giving mortality probability $q$ is

$$
x_q(s)=\frac{\operatorname{logit}(q)-\widehat\beta_0-\widehat\beta_2s}
{\widehat\beta_1+\widehat\beta_3s}.
$$

For $q=0.5$, $\operatorname{logit}(q)=0$ and the corresponding physical dose is

$$LD50(s)=2^{\bar x+s_xx_{0.5}(s)}.$$

We use the delta method on the log2-dose scale and exponentiate the confidence limits. The result is a statistical estimate, not a safe-use recommendation.

In [ ]:
beta_hat = beta_newton
hessian_hat = derivatives(beta_hat)[1]
hessian_eigenvalues = np.linalg.eigvalsh(hessian_hat)
covariance = np.linalg.inv(hessian_hat)
standard_errors = np.sqrt(np.diag(covariance))
zcrit = norm.ppf(.975)
inference = pd.DataFrame({
    "parameter": parameter_names, "estimate": beta_hat, "std_error": standard_errors,
    "lower_95": beta_hat-zcrit*standard_errors,
    "upper_95": beta_hat+zcrit*standard_errors
})
display(inference)
print("Observed-information eigenvalues:", hessian_eigenvalues)

def log2_dose_at_probability(beta, female_indicator, probability=.5):
    numerator = np.log(probability/(1-probability))-beta[0]-beta[2]*female_indicator
    denominator = beta[1]+beta[3]*female_indicator
    standardized = numerator/denominator
    return log_dose_mean + log_dose_scale*standardized

def numerical_gradient_scalar(function, beta, step=1e-6):
    result = np.empty_like(beta)
    for j in range(len(beta)):
        h = step*max(1.0, abs(beta[j]))
        forward, backward = beta.copy(), beta.copy()
        forward[j] += h; backward[j] -= h
        result[j] = (function(forward)-function(backward))/(2*h)
    return result

ld50_rows = []
for label, indicator in [("Male", 0.0), ("Female", 1.0)]:
    function = lambda b, s=indicator: log2_dose_at_probability(b, s, .5)
    estimate_log2 = function(beta_hat)
    gradient = numerical_gradient_scalar(function, beta_hat)
    se_log2 = np.sqrt(gradient @ covariance @ gradient)
    ld50_rows.append({
        "gender": label, "LD50_micrograms": 2**estimate_log2,
        "lower_95": 2**(estimate_log2-zcrit*se_log2),
        "upper_95": 2**(estimate_log2+zcrit*se_log2)
    })
ld50_table = pd.DataFrame(ld50_rows)
display(ld50_table)

## 5. Model comparison and diagnostics

The full interaction model is compared with the additive model using

$$LR=2[J_{reduced}(\widehat\beta_R)-J_{full}(\widehat\beta_F)]
\overset{H_0}{\approx}\chi_1^2.$$

Grouped-binomial Pearson residuals are

$$r_i=\frac{K_i-n_i\widehat p_i}{\sqrt{n_i\widehat p_i(1-\widehat p_i)}}.$$

We also report residual deviance, per-trial cross-entropy, Brier score, and threshold accuracy. Because this is a small designed bioassay rather than an i.i.d. prediction benchmark, random train/test splitting would discard experimental structure and is not the primary validation strategy.

In [ ]:
# Reduced model without dose-by-sex interaction.
X_reduced = X[:, :3]
def reduced_objective(beta):
    eta = X_reduced @ beta
    return float(np.sum(exposed*np.logaddexp(0, eta)-killed*eta))
def reduced_gradient(beta):
    return X_reduced.T @ (exposed*expit(X_reduced@beta)-killed)
reduced_fit = minimize(reduced_objective, np.zeros(3), jac=reduced_gradient, method="BFGS")
likelihood_ratio = 2*(reduced_fit.fun-objective(beta_hat))
lr_p_value = chi2.sf(likelihood_ratio, 1)

fitted_probability = expit(X@beta_hat)
expected_killed = exposed*fitted_probability
pearson_residual = (killed-expected_killed)/np.sqrt(
    exposed*fitted_probability*(1-fitted_probability)
)
def xlog_ratio(observed, expected):
    result = np.zeros_like(observed, dtype=float)
    positive = observed > 0
    result[positive] = observed[positive]*np.log(observed[positive]/expected[positive])
    return result
deviance = 2*np.sum(
    xlog_ratio(killed, expected_killed)
    + xlog_ratio(exposed-killed, exposed-expected_killed)
)
total_trials = exposed.sum()
brier = np.sum(killed*(1-fitted_probability)**2 +
               (exposed-killed)*fitted_probability**2)/total_trials
accuracy = np.sum(np.where(fitted_probability >= .5, killed, exposed-killed))/total_trials
cross_entropy = objective(beta_hat)/total_trials

print(f"Interaction LR statistic: {likelihood_ratio:.4f}; p-value: {lr_p_value:.4g}")
print(f"Residual deviance: {deviance:.4f} on {len(data)-len(beta_hat)} df")
print(f"Per-trial cross-entropy: {cross_entropy:.4f}")
print(f"Brier score: {brier:.4f}; threshold accuracy: {accuracy:.3f}")

dose_grid = np.geomspace(data.dose_micrograms.min(), data.dose_micrograms.max(), 250)
log2_grid = np.log2(dose_grid)
standardized_grid = (log2_grid-log_dose_mean)/log_dose_scale
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for label, indicator, color in [("Male", 0.0, "tab:blue"), ("Female", 1.0, "tab:orange")]:
    design_grid = np.column_stack([
        np.ones(len(dose_grid)), standardized_grid,
        np.full(len(dose_grid), indicator), standardized_grid*indicator
    ])
    eta = design_grid@beta_hat
    se_eta = np.sqrt(np.einsum("ij,jk,ik->i", design_grid, covariance, design_grid))
    probability = expit(eta)
    lower, upper = expit(eta-zcrit*se_eta), expit(eta+zcrit*se_eta)
    axes[0].plot(dose_grid, probability, color=color, label=label)
    axes[0].fill_between(dose_grid, lower, upper, color=color, alpha=.18)
    group = data[data.gender == label]
    axes[0].scatter(group.dose_micrograms, group.killed/group.exposed, color=color)
axes[0].set(xscale="log", xlabel="dose (micrograms; log scale)",
            ylabel="mortality probability", title="Fitted dose–response with 95% bands")
axes[0].legend()
axes[1].axhline(0, color="black", lw=1)
axes[1].scatter(expected_killed, pearson_residual,
                c=np.where(female, "tab:orange", "tab:blue"))
axes[1].set(xlabel="expected number killed", ylabel="Pearson residual",
            title="Grouped-binomial residuals")
plt.tight_layout(); plt.show()

## 6. Conclusions and limitations

- The grouped-binomial likelihood is exactly equivalent to expanding the data into 240 Bernoulli rows, apart from constants independent of $\beta$.
- The analytic gradient passes an independent finite-difference audit.
- Gradient descent, damped Newton, BFGS, and trust-exact converge to the same finite solution; Newton exploits curvature and requires far fewer iterations.
- The fitted curves quantify how mortality changes with dose and sex, while LD50 translates coefficients into an interpretable experimental quantity.
- Solver agreement and a small gradient certify a stationary numerical solution, not correct biological specification.
- Confidence intervals assume the grouped-binomial model. Extra-binomial variation, dependence, omitted experimental factors, or separation would require a richer model or penalization.

The notebook is fully local and reproducible: the official 12-row dataset is stored in `data/`, all optimization code is visible here, and assertions verify the data totals, design rank, derivative, solver agreement, and reference convergence.